# 02 · Custo e limiar

**Bloco 2 do workshop.**

Pergunta: *qual erro dói mais, e quanto?*

O limiar 0,5 é o default da biblioteca. Ninguém no negócio escolheu esse número.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# dataset canônico do workshop — NÃO altere estes parâmetros
X, y = make_classification(n_samples=20000, n_features=20, n_informative=8,
                           n_redundant=4, weights=[0.99, 0.01], flip_y=0.0,
                           class_sep=1.5, random_state=42)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
# split extra para calibrar no notebook 03 (nunca calibre no teste)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.25,
                                              stratify=y_tr, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

print(f"teste: {len(y_te)} casos | {int(y_te.sum())} fraudes | prevalência {y_te.mean():.2%}")

## 2.1 A matriz de custo

Custos assumidos neste exemplo. **Troque pelos custos do seu problema mais adiante.**

In [ ]:
C_FP = 5      # cliente legítimo bloqueado
C_FN = 500    # fraude que passou

t_teorico = C_FP / (C_FP + C_FN)
print(f"limiar teórico  t* = C_FP/(C_FP+C_FN) = {t_teorico:.4f}")
print(f"limiar padrão              = 0.5000")
print(f"razão                      = {0.5/t_teorico:.0f}x maior que o ótimo")

## 2.2 Varredura empírica do limiar

In [ ]:
import matplotlib.pyplot as plt

ths = np.linspace(0.001, 0.999, 999)
custos = np.array([C_FP*int(((proba >= t) & (y_te == 0)).sum())
                 + C_FN*int(((proba <  t) & (y_te == 1)).sum()) for t in ths])

i      = int(custos.argmin())
t_ot   = float(ths[i])
i50    = int(np.argmin(abs(ths - 0.5)))

print(f"limiar ótimo empírico: {t_ot:.4f}")
print(f"custo em t*:   R$ {custos[i]:,.0f}")
print(f"custo em 0,5:  R$ {custos[i50]:,.0f}")
print(f"redução:       {1 - custos[i]/custos[i50]:.1%}")

plt.figure(figsize=(9, 4))
plt.plot(ths, custos)
plt.axvline(t_ot, ls='--', color='crimson', label=f't* = {t_ot:.3f}')
plt.axvline(0.5, ls=':', color='gray', label='default 0,5')
plt.xlabel('limiar de decisão'); plt.ylabel('custo total (R$)')
plt.legend(); plt.tight_layout(); plt.show()

> **Por que o empírico difere do teórico?** A fórmula fechada assume custo linear e ganho zero no verdadeiro positivo. A varredura usa a distribuição real dos scores, que é discreta — a Random Forest só produz múltiplos de 1/200. O ótimo empírico é um platô, não um ponto.

## 2.3 O que muda na matriz

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

for t, nome in [(0.5, "t = 0,50 (default)"), (t_ot, f"t = {t_ot:.3f} (ótimo)")]:
    p = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, p).ravel()
    print(f"{nome}")
    print(f"   VN={tn:>5}  FP={fp:>5}  FN={fn:>4}  VP={tp:>3}"
          f"   precisão={precision_score(y_te, p):.3f}  recall={recall_score(y_te, p):.3f}")

## 2.4 ROC vs Precision-Recall

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

print(f"AUC-ROC: {roc_auc_score(y_te, proba):.4f}   (baseline 0,500)")
print(f"AUPRC:   {average_precision_score(y_te, proba):.4f}   (baseline = prevalência = {y_te.mean():.3f})")

fpr, tpr, _ = roc_curve(y_te, proba)
prec, rec, _ = precision_recall_curve(y_te, proba)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0,1],[0,1],'--',color='gray')
ax[0].set_title('ROC'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[1].plot(rec, prec); ax[1].axhline(y_te.mean(), ls='--', color='gray')
ax[1].set_title('Precision-Recall'); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precisão')
plt.tight_layout(); plt.show()

### ✏️ Tarefa — os custos do seu problema

1. Qual a razão de custo entre FN e FP? Se não houver valor em reais, use uma razão ("1 FN vale 80 FPs").
2. **Quem, fora do squad, deveria validar essa razão?** Este item é o que mais falta e o mais importante.

In [ ]:
meu_C_FP = None    # ex: 5
meu_C_FN = None    # ex: 500
quem_valida = "..."   # nome ou papel de alguém FORA do squad

if meu_C_FP and meu_C_FN:
    print(f"meu limiar teórico: {meu_C_FP/(meu_C_FP+meu_C_FN):.4f}")
print("valida:", quem_valida)